# Sparse Expert AutoSec Training (Colab)
This notebook trains adapter/expert parameters using open-source vulnerability data.


## Open-source dataset source
We use **CVEfixes** (GitHub): `https://github.com/secureIT-project/CVEfixes`.
The notebook clones the repository and also builds bootstrap samples if raw export files are unavailable.


In [ ]:
!pip -q install pandas

In [ ]:
import os, subprocess, pathlib
repo_root = pathlib.Path('/content/That-Ai-Coder')
if not repo_root.exists():
    subprocess.run(['git','clone','https://github.com/YOUR_ORG/That-Ai-Coder.git', str(repo_root)], check=False)
os.chdir(repo_root)
print('repo_root=', repo_root)


In [ ]:
from pathlib import Path
from sparse_autosec.dataset import OpenSourceDatasetLoader

loader = OpenSourceDatasetLoader(Path('/content/datasets'))
cvefixes_repo = loader.clone_cvefixes_repo()
print('CVEfixes cloned at', cvefixes_repo)
samples = loader.build_bootstrap_samples()
print('bootstrap samples=', len(samples))
print('balance=', loader.class_balance(samples))


In [ ]:
from sparse_autosec.system import SparseExpertAutoSec
from sparse_autosec.config import AutoSecConfig

cfg = AutoSecConfig()
cfg.execution.fuzz_rounds = 8
cfg.execution.mutation_rounds = 8
system = SparseExpertAutoSec(config=cfg)

for sample in samples:
    tensor_text = f'scan signature={sample.signature} code={sample.code}'
    route = system.router.route(system.core.encode_task(tensor_text), complexity=system.core.score_task_complexity(tensor_text))
    target = 1.0 if sample.label == 1 else 0.0
    system.learning.record_replay(tensor_text, route.selected[0], target)
    system.memory.counters[sample.signature] = system.memory.counters.get(sample.signature, 0) + (3 if sample.label == 1 else 0)
    if sample.label == 1 and system.learning.enter_training(sample.signature):
        metrics = system.learning.train_cycle(tensor_text, route.selected, success_target=1.0)
        system.learning.leave_training()
        print('trained', sample.signature, metrics.get('mean_loss'))


In [ ]:
import json
art = Path('/content/artifacts')
art.mkdir(exist_ok=True)
adapter_path = art / 'adapter_snapshot.json'
adapter_path.write_text(json.dumps(system.core.adapter.delta))
print('saved adapter:', adapter_path)
